# Notebook 02 — Feature engineering

Objectif : transformer les **10 tables** en **un seul** dataset (1 ligne par client), prêt pour l'entraînement.

Logique répétée pour chaque table secondaire : **one-hot encoding** des colonnes texte → **agrégation par client** (`groupby('SK_ID_CURR')`) → **jointure** `left` sur la table principale.

> **Note mémoire.** Le dataset complet est lourd (~300k lignes, 500+ colonnes). Travaille de préférence sur 16 Go de RAM. `reduce_mem_usage` allège les nombres, `gc.collect()` libère la mémoire entre les tables.

## 1. Fonctions utilitaires

In [ ]:
# =====================================================================
# Notebook 02 - Feature engineering
# Objectif : transformer les 10 tables en UN seul dataset (1 ligne/client),
#            prêt pour l'entraînement.
# Cette première cellule définit les fonctions utilitaires réutilisées
# pour chaque table.
# =====================================================================

import pandas as pd
import numpy as np
import re                  # expressions régulières (nettoyage des noms de colonnes)
import gc                  # garbage collector : libère la mémoire entre les tables


def one_hot_encoder(df, nan_as_category=True):
    """Encode les colonnes texte (catégorielles) en colonnes binaires 0/1.

    Les modèles ne comprennent que des nombres : on transforme donc chaque
    colonne de type texte en plusieurs colonnes 0/1 (une par catégorie).

    Args:
        df (DataFrame): le tableau à encoder.
        nan_as_category (bool): si True, crée aussi une colonne pour les NaN
            (l'absence d'information peut être un signal).

    Returns:
        tuple: (df encodé, liste des noms des nouvelles colonnes créées).
    """
    original_columns = list(df.columns)
    # Repérer les colonnes de type texte (dtype "object")
    categorical = [c for c in df.columns if df[c].dtype == "object"]
    # get_dummies crée une colonne 0/1 par catégorie
    df = pd.get_dummies(df, columns=categorical, dummy_na=nan_as_category)
    new_columns = [c for c in df.columns if c not in original_columns]
    return df, new_columns


def clean_column_names(df):
    """Nettoie les noms de colonnes en remplaçant tout caractère spécial par '_'.

    INDISPENSABLE avant LightGBM / XGBoost : ces librairies plantent si un nom
    de colonne contient des caractères spéciaux (virgules, crochets) hérités
    du one-hot encoding (cf. piège n°2).

    Args:
        df (DataFrame): tableau dont on nettoie les noms de colonnes.

    Returns:
        DataFrame: le même tableau avec des noms de colonnes valides et uniques.
    """
    # Remplacer toute suite de caractères non alphanumériques par un seul "_"
    df.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", str(c)) for c in df.columns]

    # Si le nettoyage a créé des doublons de noms, les rendre uniques
    if df.columns.duplicated().any():
        cols = pd.Series(df.columns)
        for dup in cols[cols.duplicated()].unique():
            idx = cols[cols == dup].index
            cols[idx] = [f"{dup}_{i}" if i else dup for i in range(len(idx))]
        df.columns = cols
    return df


def reduce_mem_usage(df):
    """Réduit l'empreinte mémoire en abaissant la précision des nombres.

    Convertit les float64 -> float32 et réduit les entiers au plus petit type
    possible. Utile car le dataset final est volumineux (~300k lignes, 500+ col.).

    Args:
        df (DataFrame): tableau à alléger.

    Returns:
        DataFrame: le même tableau, plus léger en mémoire.
    """
    for col in df.columns:
        dtype = df[col].dtype
        if dtype == "float64":
            df[col] = df[col].astype("float32")
        elif dtype == "int64":
            df[col] = pd.to_numeric(df[col], downcast="integer")
    return df

## 2. Table principale `application`

On nettoie deux anomalies (genre « XNA », valeur `365243`) et on crée des **ratios métier** très discriminants.

In [ ]:
def process_application(path):
    """Charge et nettoie la table principale, et crée des ratios métier.

    Args:
        path (str): chemin du dossier contenant les CSV (ex. "../data/raw/").

    Returns:
        DataFrame: la table principale nettoyée, encodée, 1 ligne par client.
    """
    df = pd.read_csv(path + "application_train.csv")

    # Anomalie de genre : quelques lignes "XNA" (valeur invalide) -> on les retire
    df = df[df["CODE_GENDER"] != "XNA"]

    # Anomalie DAYS_EMPLOYED (365243 = non renseigné) -> NaN  (piège n°1)
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

    # --- Features métier (ratios) : très discriminantes pour le crédit ---
    df["DAYS_EMPLOYED_PERC"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]       # ancienneté emploi / âge
    df["INCOME_CREDIT_PERC"] = df["AMT_INCOME_TOTAL"] / df["AMT_CREDIT"]    # revenu / crédit
    df["INCOME_PER_PERSON"]  = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]   # revenu par personne
    df["ANNUITY_INCOME_PERC"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]  # part du revenu = mensualité
    df["PAYMENT_RATE"]       = df["AMT_ANNUITY"] / df["AMT_CREDIT"]         # mensualité / crédit

    df, _ = one_hot_encoder(df)
    return df

## 3. `bureau` + `bureau_balance`

Double agrégation : historique mensuel par crédit externe (`SK_ID_BUREAU`), puis tous les crédits par client (`SK_ID_CURR`).

In [ ]:
def process_bureau_and_balance(path):
    """Agrège les crédits externes (bureau) et leur historique (bureau_balance).

    Deux niveaux d'agrégation : d'abord l'historique mensuel par crédit
    (SK_ID_BUREAU), puis tous les crédits par client (SK_ID_CURR).

    Args:
        path (str): chemin du dossier des CSV.

    Returns:
        DataFrame: features agrégées par client (préfixe "BURO_").
    """
    bureau = pd.read_csv(path + "bureau.csv")
    bb = pd.read_csv(path + "bureau_balance.csv")

    # 1) Agréger l'historique mensuel par crédit externe (clé SK_ID_BUREAU)
    bb, bb_cat = one_hot_encoder(bb)
    bb_aggregations = {"MONTHS_BALANCE": ["min", "max", "size"]}
    for col in bb_cat:
        bb_aggregations[col] = ["mean"]
    bb_agg = bb.groupby("SK_ID_BUREAU").agg(bb_aggregations)
    bb_agg.columns = pd.Index([e[0] + "_" + e[1].upper() for e in bb_agg.columns])

    bureau = bureau.join(bb_agg, how="left", on="SK_ID_BUREAU")
    bureau.drop(columns=["SK_ID_BUREAU"], inplace=True)
    del bb, bb_agg; gc.collect()

    # 2) Agréger les crédits externes par client (clé SK_ID_CURR)
    bureau, bureau_cat = one_hot_encoder(bureau)
    num_aggregations = {
        "DAYS_CREDIT": ["mean", "var", "min", "max"],
        "DAYS_CREDIT_ENDDATE": ["mean", "max"],
        "AMT_CREDIT_SUM": ["mean", "sum", "max"],
        "AMT_CREDIT_SUM_DEBT": ["mean", "sum"],
        "AMT_CREDIT_SUM_OVERDUE": ["mean"],
        "CREDIT_DAY_OVERDUE": ["mean", "max"],
        "AMT_ANNUITY": ["mean", "max"],
    }
    cat_aggregations = {c: ["mean"] for c in bureau_cat}
    bureau_agg = bureau.groupby("SK_ID_CURR").agg({**num_aggregations, **cat_aggregations})
    bureau_agg.columns = pd.Index(["BURO_" + e[0] + "_" + e[1].upper() for e in bureau_agg.columns])
    bureau_agg["BURO_COUNT"] = bureau.groupby("SK_ID_CURR").size()   # nb de crédits externes
    del bureau; gc.collect()
    return bureau_agg

## 4. `previous_application`

In [ ]:
def process_previous_application(path):
    """Agrège les demandes de crédit précédentes chez Prêt à dépenser.

    Args:
        path (str): chemin du dossier des CSV.

    Returns:
        DataFrame: features agrégées par client (préfixe "PREV_").
    """
    prev = pd.read_csv(path + "previous_application.csv")
    prev, cat_cols = one_hot_encoder(prev)

    # Neutraliser la valeur aberrante 365243 (= "non renseigné") dans les DAYS_* (piège n°1)
    for col in ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION",
                "DAYS_LAST_DUE", "DAYS_TERMINATION"]:
        prev[col] = prev[col].replace(365243, np.nan)

    # Ratio montant demandé / montant accordé (signal de négociation passée)
    prev["APP_CREDIT_PERC"] = prev["AMT_APPLICATION"] / prev["AMT_CREDIT"]

    num_aggregations = {
        "AMT_ANNUITY": ["mean", "max"],
        "AMT_APPLICATION": ["mean", "max"],
        "AMT_CREDIT": ["mean", "max"],
        "APP_CREDIT_PERC": ["mean", "max"],
        "AMT_DOWN_PAYMENT": ["mean"],
        "DAYS_DECISION": ["mean", "max"],
        "CNT_PAYMENT": ["mean", "sum"],
    }
    cat_aggregations = {c: ["mean"] for c in cat_cols}
    prev_agg = prev.groupby("SK_ID_CURR").agg({**num_aggregations, **cat_aggregations})
    prev_agg.columns = pd.Index(["PREV_" + e[0] + "_" + e[1].upper() for e in prev_agg.columns])
    prev_agg["PREV_COUNT"] = prev.groupby("SK_ID_CURR").size()
    del prev; gc.collect()
    return prev_agg

## 5. `POS_CASH_balance`

In [ ]:
def process_pos_cash(path):
    """Agrège l'historique mensuel des crédits POS (point de vente) et cash.

    Args:
        path (str): chemin du dossier des CSV.

    Returns:
        DataFrame: features agrégées par client (préfixe "POS_").
    """
    pos = pd.read_csv(path + "POS_CASH_balance.csv")
    pos, cat_cols = one_hot_encoder(pos)

    aggregations = {
        "MONTHS_BALANCE": ["max", "mean", "size"],
        "SK_DPD": ["max", "mean"],          # jours de retard (Days Past Due)
        "SK_DPD_DEF": ["max", "mean"],      # jours de retard avec défaut
    }
    for col in cat_cols:
        aggregations[col] = ["mean"]
    pos_agg = pos.groupby("SK_ID_CURR").agg(aggregations)
    pos_agg.columns = pd.Index(["POS_" + e[0] + "_" + e[1].upper() for e in pos_agg.columns])
    pos_agg["POS_COUNT"] = pos.groupby("SK_ID_CURR").size()
    del pos; gc.collect()
    return pos_agg

## 6. `installments_payments`

Le comportement de paiement passé (retards, montants payés) est l'un des meilleurs prédicteurs du défaut.

In [ ]:
def process_installments(path):
    """Agrège l'historique des échéances/paiements des crédits précédents.

    Construit des features de comportement de paiement (retards, montants payés)
    qui comptent parmi les meilleurs prédicteurs du défaut futur.

    Args:
        path (str): chemin du dossier des CSV.

    Returns:
        DataFrame: features agrégées par client (préfixe "INS_").
    """
    ins = pd.read_csv(path + "installments_payments.csv")

    # Features de comportement de paiement
    ins["PAYMENT_PERC"] = ins["AMT_PAYMENT"] / ins["AMT_INSTALMENT"]   # % payé de l'échéance
    ins["PAYMENT_DIFF"] = ins["AMT_INSTALMENT"] - ins["AMT_PAYMENT"]   # reste dû
    # Retard (DPD) et avance (DBD), bornés à 0
    ins["DPD"] = (ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]).clip(lower=0)
    ins["DBD"] = (ins["DAYS_INSTALMENT"] - ins["DAYS_ENTRY_PAYMENT"]).clip(lower=0)

    ins, cat_cols = one_hot_encoder(ins)
    aggregations = {
        "PAYMENT_PERC": ["mean", "max", "min"],
        "PAYMENT_DIFF": ["mean", "max"],
        "DPD": ["mean", "max"],
        "DBD": ["mean", "max"],
        "AMT_PAYMENT": ["mean", "sum"],
    }
    for col in cat_cols:
        aggregations[col] = ["mean"]
    ins_agg = ins.groupby("SK_ID_CURR").agg(aggregations)
    ins_agg.columns = pd.Index(["INS_" + e[0] + "_" + e[1].upper() for e in ins_agg.columns])
    ins_agg["INS_COUNT"] = ins.groupby("SK_ID_CURR").size()
    del ins; gc.collect()
    return ins_agg

## 7. `credit_card_balance`

In [ ]:
def process_credit_card(path):
    """Agrège l'historique mensuel des cartes de crédit précédentes.

    Args:
        path (str): chemin du dossier des CSV.

    Returns:
        DataFrame: features agrégées par client (préfixe "CC_").
    """
    cc = pd.read_csv(path + "credit_card_balance.csv")
    cc, _ = one_hot_encoder(cc)
    cc.drop(columns=["SK_ID_PREV"], inplace=True)   # identifiant inutile pour l'agrégation

    cc_agg = cc.groupby("SK_ID_CURR").agg(["mean", "max", "min", "sum", "var"])
    cc_agg.columns = pd.Index(["CC_" + e[0] + "_" + e[1].upper() for e in cc_agg.columns])
    cc_agg["CC_COUNT"] = cc.groupby("SK_ID_CURR").size()
    del cc; gc.collect()
    return cc_agg

## 8. Assemblage final et sauvegarde

On part de la table principale et on colle chaque table agrégée. Le `assert` protège contre la duplication de lignes (l'erreur de jointure la plus dangereuse).

In [ ]:
# --- Assemblage : on part de la table principale et on colle chaque table agrégée ---
PATH = "../data/raw/"   # dossier des CSV bruts

# Table de départ : 1 ligne par client, avec la cible et les ratios métier
df = process_application(PATH)
print("application :", df.shape)

# Pour chaque table secondaire : on calcule ses features agrégées puis on les
# joint à la table principale par l'identifiant client SK_ID_CURR.
for name, func in [
    ("bureau", process_bureau_and_balance),
    ("previous", process_previous_application),
    ("pos", process_pos_cash),
    ("installments", process_installments),
    ("credit_card", process_credit_card),
]:
    agg = func(PATH)
    df = df.join(agg, how="left", on="SK_ID_CURR")
    # Contrôle de sécurité : le nombre de lignes ne doit JAMAIS changer
    assert df.shape[0] == df["SK_ID_CURR"].nunique(), "Jointure a dupliqué des lignes !"
    print(f"après {name:12s} : {df.shape}")
    del agg; gc.collect()

# Nettoyage final des noms de colonnes (pour LightGBM/XGBoost, piège n°2)
df = clean_column_names(df)
df = reduce_mem_usage(df)

print("Dataset final :", df.shape)
print("Répartition cible :", df["TARGET"].value_counts(normalize=True).round(3).to_dict())

# Sauvegarde au format Parquet (rapide, compressé) -> rechargé au notebook 03
df.to_parquet("../data/processed/dataset_final.parquet")
print("Sauvegardé dans data/processed/dataset_final.parquet")

## 9. Sauvegarde Git (fin du notebook 02)

```powershell
git add notebooks/02_feature_engineering.ipynb
git commit -m "Notebook 02 : feature engineering, fusion des 10 tables"
git push
```

> Le fichier `dataset_final.parquet` n'est **pas** versionné (exclu par `.gitignore`, trop volumineux). On versionne le **code** qui le génère.